# 03 · Results analysis

Loads every finished experiment and compares modes. Nothing here is pre-computed: run the experiments first (`python scripts/run_experiments.py`).

> Results produced on the synthetic dataset only validate the pipeline and must not be reported as findings.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
import ppfl  # noqa: F401  (Windows OpenMP import-order guard)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ppfl.visualization.plots import set_style
set_style()
%matplotlib inline

In [ ]:
from ppfl.evaluation.comparison import collect_experiments
RESULTS = ROOT / "results" / "experiments"   # or ROOT / "results" / "synthetic" / "experiments"
summary, histories = collect_experiments(RESULTS)
if summary.empty:
    print("No finished experiments yet. Results will appear after running the experiments.")
summary

## Headline table — unseen-attack recall next to the false-positive rate

In [ ]:
cols = ["display", "dataset", "unseen_recall", "known_recall", "f1", "precision", "recall", "fpr", "roc_auc", "epsilon", "total_mb", "train_time_s"]
summary[[c for c in cols if c in summary]].sort_values("unseen_recall", ascending=False).round(4) if not summary.empty else None

## Convergence

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for name, h in histories.items():
    if "f1" in h:
        axes[0].plot(h["round"], h["f1"], label=name)
    if "val_loss" in h:
        axes[1].plot(h["round"], h["val_loss"], label=name)
axes[0].set(title="F1 vs round / epoch", xlabel="Round", ylabel="F1")
axes[1].set(title="Validation loss", xlabel="Round", ylabel="MSE", yscale="log")
axes[0].legend(fontsize=7)

## Privacy / utility trade-off

In [ ]:
dp = summary[summary["epsilon"].notna()].sort_values("epsilon") if not summary.empty else pd.DataFrame()
if len(dp):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(dp["epsilon"], dp["f1"], marker="o", label="F1")
    ax.plot(dp["epsilon"], dp["unseen_recall"], marker="s", label="Unseen-attack recall")
    ax.set_xscale("log"); ax.set(xlabel="ε", ylabel="Value", title="Utility vs privacy"); ax.legend()
else:
    print("No DP experiments found.")

## Per-class detection rates of one experiment

In [ ]:
from ppfl.utils.serialization import load_json
EXPERIMENT = "full_ppfl"
path = RESULTS / EXPERIMENT / "metrics.json"
if path.exists():
    report = load_json(path)["models"]["autoencoder"]["report"]
    display(pd.DataFrame(report["per_class"]).T.sort_values("group"))
else:
    print(f"{path} not found")